# 🤖 NewsBot Intelligence System 2.0 — Final Project Implementation
### ITAI 2373 · Natural Language Processing

A production-ready news-intelligence platform that upgrades the midterm NewsBot into a full
advanced-NLP system covering the four required core modules:

1. **Advanced Content Analysis Engine** — enhanced classification (confidence + explanation), topic modeling, sentiment evolution, entity-relationship knowledge graph
2. **Language Understanding & Generation** — extractive/abstractive summarization, semantic search, content enhancement
3. **Multilingual Intelligence** — language detection, translation, cross-lingual analysis
4. **Conversational Interface** — intent classification, slot filling, context-aware responses

**Design principle — graceful degradation.** Every component runs on a reliable backbone
(`scikit-learn` + `spaCy` + `NLTK` + `networkx`) and *automatically upgrades* if heavier
libraries (`transformers`, `sentence-transformers`, `gensim`, `deep-translator`) are installed.
The notebook therefore runs anywhere — from a lean Jupyter kernel to a full Colab GPU runtime —
without code changes.

> **How to run:** place `BBC News Train.csv` (Kaggle *learn-ai-bbc*) beside this notebook and
> run all cells top-to-bottom. Optional power-ups: `pip install transformers sentence-transformers deep-translator gensim`.

## 🏗️ Section 1 — Project Setup & Architecture

**Architecture.** NewsBot 2.0 uses a *modular* design: each capability is an independent,
testable class with a single responsibility, and a top-level orchestrator
(`NewsBot2IntegratedSystem`) wires them together and manages shared state.

```
                    ┌─────────────────────────────┐
   raw article ───▶ │  NewsBot2IntegratedSystem    │
   / user query     │  (orchestrator + shared state)│
                    └──────────────┬───────────────┘
      ┌──────────────┬─────────────┼──────────────┬───────────────┐
      ▼              ▼             ▼              ▼               ▼
 Classifier   TopicEngine   Sentiment/NER   Summarizer/Search   Multilingual
      └──────────────┴─────────────┴──────────────┴───────────────┘
                                   ▼
                        ConversationalInterface  ◀── natural-language queries
```

**Data flow:** load → normalise language → preprocess → analyse (classify / topics / sentiment /
entities / summary / search) → enhance → respond.  Errors are contained per-component so one
failure never cascades (a lesson carried from the midterm's `FileNotFoundError` → `NameError`
chain and its `MaxAbsScaler` feature-scaling fix).

The cell below sets up imports and **detects capabilities** so the rest of the notebook adapts
to whatever is installed.

In [1]:
# ============================================================
# NewsBot 2.0 — Part 1: Setup, Config, Data, Preprocessing
# ============================================================
import os, re, glob, json, warnings, math
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# ---- Capability detection (optional heavy libs upgrade the system if present) ----
def _try(name):
    try:
        __import__(name); return True
    except Exception:
        return False

HAS_SPACY   = _try("spacy")
HAS_NLTK    = _try("nltk")
HAS_GENSIM  = _try("gensim")
HAS_LANGDET = _try("langdetect")
HAS_TRANSFORMERS = _try("transformers")
HAS_SBERT   = _try("sentence_transformers")
HAS_TRANSLATOR = _try("deep_translator")

CAPS = dict(spacy=HAS_SPACY, nltk=HAS_NLTK, gensim=HAS_GENSIM,
            langdetect=HAS_LANGDET, transformers=HAS_TRANSFORMERS,
            sentence_transformers=HAS_SBERT, deep_translator=HAS_TRANSLATOR)

# ---- NLP resource bootstrap ----
NLP = None
if HAS_SPACY:
    import spacy
    try:
        NLP = spacy.load("en_core_web_sm")
    except Exception:
        NLP = None

if HAS_NLTK:
    import nltk
    for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet",
                "vader_lexicon", "averaged_perceptron_tagger",
                "averaged_perceptron_tagger_eng"]:
        try: nltk.data.find(pkg)
        except Exception:
            try: nltk.download(pkg, quiet=True)
            except Exception: pass
    from nltk.corpus import stopwords as _sw
    try: STOPWORDS = set(_sw.words("english"))
    except Exception: STOPWORDS = set()
    from nltk.stem import WordNetLemmatizer
    _LEMMA = WordNetLemmatizer()
else:
    STOPWORDS = set("a an the and or but of to in on for with at by from is are was were be been "
                    "this that these those it its as not no we you they he she".split())
    _LEMMA = None


class NewsBot2Config:
    """Central configuration for every NewsBot 2.0 component."""
    def __init__(self):
        # Data
        self.data_candidates = ["BBC News Train.csv", "bbc-text.csv", "BBC_News_Train.csv",
                                "bbc_news.csv", "news.csv"]
        self.text_col_candidates = ["Text", "text", "content", "article", "body"]
        self.label_col_candidates = ["Category", "category", "label", "target", "topic"]
        self.id_col_candidates = ["ArticleId", "id", "ArticleID"]
        # Modelling
        self.tfidf_max_features = 5000
        self.tfidf_ngram_range = (1, 2)
        self.test_size = 0.2
        self.random_state = 42
        self.n_topics = 5
        self.topic_method = "lda"          # 'lda' | 'nmf'
        self.embedding_method = "auto"     # 'auto' -> sbert if available else lsa
        self.summary_sentences = 3
        self.sentiment_anomaly_z = 2.0
        self.default_target_language = "en"
        # Runtime capabilities
        self.caps = CAPS
    def __repr__(self):
        on = [k for k, v in self.caps.items() if v]
        return f"<NewsBot2Config topics={self.n_topics} method={self.topic_method} caps_on={on}>"


def load_news_data(config, verbose=True):
    """Multi-schema loader: auto-detects the BBC file & normalises columns to
    ArticleId / Text / Category. Falls back to any csv present. Carries the
    midterm 'smart loader' philosophy so format drift never breaks the pipeline."""
    path = None
    for c in config.data_candidates:
        hits = glob.glob(c) + glob.glob(os.path.join("**", c), recursive=True)
        if hits:
            path = hits[0]; break
    if path is None:
        anycsv = glob.glob("*.csv") + glob.glob(os.path.join("**", "*.csv"), recursive=True)
        if anycsv: path = anycsv[0]
    if path is None:
        raise FileNotFoundError("No dataset found. Place 'BBC News Train.csv' next to the notebook.")

    df = pd.read_csv(path)
    def _pick(cands, cols):
        for c in cands:
            if c in cols: return c
        return None
    cols = list(df.columns)
    tcol = _pick(config.text_col_candidates, cols)
    lcol = _pick(config.label_col_candidates, cols)
    icol = _pick(config.id_col_candidates, cols)
    if tcol is None:
        # widest object column = text
        obj = [c for c in cols if df[c].dtype == object]
        tcol = max(obj, key=lambda c: df[c].astype(str).str.len().mean()) if obj else cols[0]
    out = pd.DataFrame()
    out["ArticleId"] = df[icol] if icol else range(len(df))
    out["Text"] = df[tcol].astype(str)
    out["Category"] = df[lcol].astype(str) if lcol else "unknown"
    out = out.dropna(subset=["Text"]).reset_index(drop=True)
    if verbose:
        print(f"Loaded {len(out)} articles from '{path}'")
        print(f"  columns mapped: text='{tcol}', label='{lcol}', id='{icol}'")
        if lcol: print(f"  categories: {out['Category'].value_counts().to_dict()}")
    return out


def preprocess_text(text, lemmatize=True, keep_stopwords=False):
    """Clean -> lowercase -> tokenise -> (optional) stopword removal + lemmatise."""
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    if not keep_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 2]
    if lemmatize and _LEMMA is not None:
        tokens = [_LEMMA.lemmatize(t) for t in tokens]
    return " ".join(tokens)


In [2]:
# Quick sanity check: configuration + capabilities + data
config = NewsBot2Config()
print(config)
print("Capabilities online:", {k: v for k, v in config.caps.items() if v})

df = load_news_data(config)
df["clean_text"] = df["Text"].map(preprocess_text)
df.head(3)

<NewsBot2Config topics=5 method=lda caps_on=['spacy', 'nltk', 'gensim', 'langdetect']>
Capabilities online: {'spacy': True, 'nltk': True, 'gensim': True, 'langdetect': True}
Loaded 200 articles from 'BBC News Train.csv'
  columns mapped: text='Text', label='Category', id='ArticleId'
  categories: {'sport': 40, 'entertainment': 40, 'politics': 40, 'business': 40, 'tech': 40}


,ArticleId,Text,Category,clean_text
0,1094,The striker scored a stunning goal in the fina...,sport,striker scored stunning goal final minute secu...
1,1182,The streaming service renewed the popular dram...,entertainment,streaming service renewed popular drama series...
2,1112,The athletics team broke a world record at the...,sport,athletics team broke world record championship...


## 📊 Section 2 — Advanced Content Analysis Engine

Four upgraded analyzers:

| Class | Upgrade over midterm |
|---|---|
| `AdvancedNewsClassifier` | soft-voting **ensemble** (LogReg + calibrated LinearSVC + NB), **confidence scores**, alternative classes, and **`explain_prediction`** (top contributing TF-IDF terms) |
| `TopicDiscoveryEngine` | unsupervised **LDA/NMF** topic discovery, auto topic labels, per-article distribution, **coherence** scoring, and **temporal trend** tracking |
| `SentimentEvolutionTracker` | VADER (or transformer) sentiment with **time-series aggregation** and **z-score anomaly detection** |
| `EntityRelationshipMapper` | spaCy NER + **relationship extraction** + a **networkx knowledge graph** with entity-connection queries |

In [3]:
# ============================================================
# NewsBot 2.0 — Part 2: Advanced Content Analysis Engine
# ============================================================
import numpy as np
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF, TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
import networkx as nx



# ---------------------------------------------------------------
# 1. Enhanced classification with confidence + explanation
# ---------------------------------------------------------------
class AdvancedNewsClassifier:
    """Ensemble TF-IDF classifier with calibrated confidence, alternative-class
    scores, and feature-based explanation. Upgrades the midterm single model."""
    def __init__(self, config=None):
        self.config = config or NewsBot2Config()
        self.vectorizer = TfidfVectorizer(max_features=self.config.tfidf_max_features,
                                          ngram_range=self.config.tfidf_ngram_range,
                                          sublinear_tf=True)
        # Calibrated members so we get real probabilities from all of them
        lr = LogisticRegression(max_iter=1000, C=5.0)
        svc = CalibratedClassifierCV(LinearSVC(C=1.0))
        nb = MultinomialNB()
        self.model = VotingClassifier(
            estimators=[("lr", lr), ("svc", svc), ("nb", nb)], voting="soft")
        self.classes_ = None
        self._fitted = False

    def train(self, texts, labels):
        X = self.vectorizer.fit_transform([preprocess_text(t) for t in texts])
        self.model.fit(X, labels)
        self.classes_ = self.model.classes_
        self._fitted = True
        return self

    def predict_with_confidence(self, article_text, top_k=3):
        assert self._fitted, "call train() first"
        X = self.vectorizer.transform([preprocess_text(article_text)])
        probs = self.model.predict_proba(X)[0]
        order = np.argsort(probs)[::-1]
        ranked = [(self.classes_[i], float(probs[i])) for i in order]
        return {
            "primary": ranked[0][0],
            "confidence": ranked[0][1],
            "alternatives": ranked[1:top_k],
            "is_confident": ranked[0][1] >= 0.6,
        }

    def explain_prediction(self, article_text, top_n=8):
        """Surface the TF-IDF terms in the article that most pull toward the
        predicted class (uses the logistic-regression member's coefficients)."""
        pred = self.predict_with_confidence(article_text)["primary"]
        X = self.vectorizer.transform([preprocess_text(article_text)])
        feats = np.array(self.vectorizer.get_feature_names_out())
        present = X.toarray()[0]
        try:
            lr = dict(self.model.named_estimators_)["lr"]
            cls_idx = list(lr.classes_).index(pred)
            weights = lr.coef_[cls_idx]
            contrib = present * weights
            idx = np.argsort(contrib)[::-1]
            top = [(feats[i], round(float(contrib[i]), 4)) for i in idx if present[i] > 0][:top_n]
        except Exception:
            idx = np.argsort(present)[::-1]
            top = [(feats[i], round(float(present[i]), 4)) for i in idx if present[i] > 0][:top_n]
        return {"prediction": pred, "top_terms": top}

    def evaluate(self, texts, labels):
        X = self.vectorizer.transform([preprocess_text(t) for t in texts])
        preds = self.model.predict(X)
        return {
            "accuracy": round(accuracy_score(labels, preds), 4),
            "macro_f1": round(f1_score(labels, preds, average="macro"), 4),
            "report": classification_report(labels, preds, zero_division=0),
        }


# ---------------------------------------------------------------
# 2. Topic modelling (LDA / NMF) with coherence + trends
# ---------------------------------------------------------------
class TopicDiscoveryEngine:
    """Unsupervised topic discovery with LDA or NMF, topic labelling, per-article
    topic distribution, coherence scoring (gensim if present), and time trends."""
    def __init__(self, n_topics=5, method="lda", config=None):
        self.config = config or NewsBot2Config()
        self.n_topics = n_topics
        self.method = method
        self.vectorizer = None
        self.model = None
        self.feature_names = None
        self.topic_labels = {}

    def fit_topics(self, documents):
        clean = [preprocess_text(d) for d in documents]
        if self.method == "nmf":
            self.vectorizer = TfidfVectorizer(max_features=2000, ngram_range=(1, 2))
            dtm = self.vectorizer.fit_transform(clean)
            self.model = NMF(n_components=self.n_topics, random_state=42, init="nndsvda", max_iter=400)
        else:
            self.vectorizer = CountVectorizer(max_features=2000, ngram_range=(1, 1))
            dtm = self.vectorizer.fit_transform(clean)
            self.model = LatentDirichletAllocation(n_components=self.n_topics,
                                                   random_state=42, learning_method="batch")
        self.model.fit(dtm)
        self.feature_names = self.vectorizer.get_feature_names_out()
        for t, comp in enumerate(self.model.components_):
            top = [self.feature_names[i] for i in comp.argsort()[::-1][:6]]
            self.topic_labels[t] = ", ".join(top[:3])
        self._clean_cache = clean
        return self

    def top_words(self, topic_id, n=8):
        comp = self.model.components_[topic_id]
        return [self.feature_names[i] for i in comp.argsort()[::-1][:n]]

    def get_article_topics(self, article_text):
        dtm = self.vectorizer.transform([preprocess_text(article_text)])
        dist = self.model.transform(dtm)[0]
        dist = dist / (dist.sum() + 1e-9)
        top = int(np.argmax(dist))
        return {"dominant_topic": top, "label": self.topic_labels[top],
                "distribution": {int(i): round(float(p), 3) for i, p in enumerate(dist)}}

    def coherence(self, documents=None):
        """u_mass-free coherence via gensim c_v if available, else avg pairwise
        term co-occurrence proxy."""
        if HAS_GENSIM:
            try:
                from gensim.corpora import Dictionary
                from gensim.models import CoherenceModel
                toks = [preprocess_text(d).split() for d in (documents or self._doc_cache_texts())]
                topics = [self.top_words(t, 10) for t in range(self.n_topics)]
                d = Dictionary(toks)
                cm = CoherenceModel(topics=topics, texts=toks, dictionary=d, coherence="c_v")
                return round(float(cm.get_coherence()), 4)
            except Exception:
                pass
        return None

    def _doc_cache_texts(self):
        return getattr(self, "_clean_cache", [])

    def track_topic_trends(self, documents, dates):
        """Aggregate dominant-topic counts per period to reveal rise/decline."""
        import pandas as pd
        rows = []
        for doc, dt in zip(documents, dates):
            info = self.get_article_topics(doc)
            rows.append({"date": pd.to_datetime(dt), "topic": info["dominant_topic"]})
        tdf = pd.DataFrame(rows)
        tdf["period"] = tdf["date"].dt.to_period("M").astype(str)
        trend = tdf.groupby(["period", "topic"]).size().unstack(fill_value=0)
        return trend


# ---------------------------------------------------------------
# 3. Sentiment evolution + anomaly detection
# ---------------------------------------------------------------
class SentimentEvolutionTracker:
    """VADER-based sentiment (optional transformer upgrade) with temporal
    aggregation and z-score anomaly detection."""
    def __init__(self, config=None):
        self.config = config or NewsBot2Config()
        self._vader = None
        if HAS_NLTK:
            try:
                from nltk.sentiment import SentimentIntensityAnalyzer
                self._vader = SentimentIntensityAnalyzer()
            except Exception:
                self._vader = None

    def analyze_sentiment(self, article_text):
        if self._vader is not None:
            s = self._vader.polarity_scores(str(article_text))
            label = ("positive" if s["compound"] >= 0.05
                     else "negative" if s["compound"] <= -0.05 else "neutral")
            return {"label": label, "compound": s["compound"],
                    "pos": s["pos"], "neu": s["neu"], "neg": s["neg"]}
        # lexicon-free fallback
        pos = sum(w in str(article_text).lower() for w in ["win", "gain", "rise", "success", "record"])
        neg = sum(w in str(article_text).lower() for w in ["loss", "fall", "cut", "crisis", "fail"])
        comp = (pos - neg) / (pos + neg + 1)
        return {"label": "positive" if comp > 0 else "negative" if comp < 0 else "neutral",
                "compound": comp, "pos": pos, "neu": 0, "neg": neg}

    def track_sentiment_over_time(self, documents, dates):
        import pandas as pd
        rows = [{"date": pd.to_datetime(d), "compound": self.analyze_sentiment(t)["compound"]}
                for t, d in zip(documents, dates)]
        sdf = pd.DataFrame(rows)
        sdf["period"] = sdf["date"].dt.to_period("M").astype(str)
        return sdf.groupby("period")["compound"].mean()

    def detect_sentiment_anomalies(self, sentiment_timeline):
        vals = np.asarray(sentiment_timeline.values, dtype=float)
        mu, sd = vals.mean(), vals.std() + 1e-9
        z = (vals - mu) / sd
        thr = self.config.sentiment_anomaly_z
        return {str(sentiment_timeline.index[i]): round(float(z[i]), 2)
                for i in range(len(vals)) if abs(z[i]) >= thr}


# ---------------------------------------------------------------
# 4. Entity relationship mapping + knowledge graph
# ---------------------------------------------------------------
class EntityRelationshipMapper:
    """spaCy NER with defensive guards, co-occurrence relationship extraction,
    and a networkx knowledge graph supporting entity-connection queries."""
    def __init__(self, config=None):
        self.config = config or NewsBot2Config()
        self.graph = nx.Graph()

    def extract_entities(self, article_text):
        if NLP is None:
            return []                      # defensive: no spaCy -> empty
        doc = NLP(str(article_text)[:100000])
        ents = [(e.text.strip(), e.label_) for e in doc.ents if e.text.strip()]
        return ents

    def extract_relationships(self, article_text):
        """Co-occurrence within an article => undirected relationship, typed by
        the entity-label pair (e.g. PERSON-ORG)."""
        ents = self.extract_entities(article_text)
        uniq = list({(t, l) for t, l in ents})
        rels = []
        for i in range(len(uniq)):
            for j in range(i + 1, len(uniq)):
                (t1, l1), (t2, l2) = uniq[i], uniq[j]
                rels.append((t1, t2, f"{l1}-{l2}"))
        return rels

    def build_knowledge_graph(self, articles):
        self.graph = nx.Graph()
        for art in articles:
            for t1, t2, rtype in self.extract_relationships(art):
                if self.graph.has_edge(t1, t2):
                    self.graph[t1][t2]["weight"] += 1
                else:
                    self.graph.add_edge(t1, t2, weight=1, rtype=rtype)
        return self.graph

    def find_entity_connections(self, e1, e2):
        if e1 not in self.graph or e2 not in self.graph:
            return None
        try:
            path = nx.shortest_path(self.graph, e1, e2)
            return {"path": path, "degree": len(path) - 1}
        except nx.NetworkXNoPath:
            return {"path": None, "degree": None}

    def key_players(self, top_n=10):
        if self.graph.number_of_nodes() == 0:
            return []
        cent = nx.degree_centrality(self.graph)
        return sorted(cent.items(), key=lambda x: x[1], reverse=True)[:top_n]


In [4]:
# --- Section 2 demo ---
from sklearn.model_selection import train_test_split
Xtr, Xte, ytr, yte = train_test_split(df["Text"], df["Category"],
                                      test_size=config.test_size,
                                      random_state=config.random_state,
                                      stratify=df["Category"])

# 1) Classification with confidence + explanation
clf = AdvancedNewsClassifier(config).train(Xtr.tolist(), ytr.tolist())
print("Eval:", {k: v for k, v in clf.evaluate(Xte.tolist(), yte.tolist()).items() if k != "report"})
demo = "Shares rose sharply after the bank reported record quarterly profits."
print("Predict:", clf.predict_with_confidence(demo))
print("Explain:", clf.explain_prediction(demo)["top_terms"][:5])

# 2) Topic modeling
topics = TopicDiscoveryEngine(config.n_topics, config.topic_method, config).fit_topics(df["Text"].tolist())
for t in range(config.n_topics):
    print(f"  Topic {t}: {', '.join(topics.top_words(t, 6))}")
print("Coherence (c_v):", topics.coherence(df["Text"].tolist()))

# 3) Sentiment
senti = SentimentEvolutionTracker(config)
print("Sentiment:", senti.analyze_sentiment(demo))

# 4) Entity knowledge graph
ents = EntityRelationshipMapper(config)
g = ents.build_knowledge_graph(df["Text"].tolist())
print(f"Knowledge graph: {g.number_of_nodes()} entities, {g.number_of_edges()} relationships")
print("Key players:", ents.key_players(5))

Eval: {'accuracy': 1.0, 'macro_f1': 1.0}
Predict: {'primary': np.str_('business'), 'confidence': 0.6855593765787468, 'alternatives': [(np.str_('sport'), 0.08900201698251443), (np.str_('entertainment'), 0.08586266621840233)], 'is_confident': True}
Explain: [('share', 0.3422), ('record', 0.3209), ('reported', 0.3148), ('sharply', 0.3148), ('profit', 0.3148)]


  Topic 0: major, year, later, actor, premiere, role
  Topic 1: match, team, championship, world, set, straight
  Topic 2: new, battery, region, designed, across, engineer
  Topic 3: party, new, ahead, prime, minister, tax
  Topic 4: bank, market, cost, rising, announced, deal
Coherence (c_v): 0.9216
Sentiment: {'label': 'positive', 'compound': 0.6249, 'pos': 0.389, 'neu': 0.611, 'neg': 0.0}


Knowledge graph: 9 entities, 10 relationships
Key players: [('annual', 0.375), ('next year', 0.375), ('quarterly', 0.375), ('billions', 0.375), ('another season', 0.25)]


## 🧠 Section 3 — Language Understanding & Generation

| Class | What it does |
|---|---|
| `IntelligentSummarizer` | **extractive TextRank** (PageRank over sentence-similarity graph) always available; **abstractive** BART path auto-enables with `transformers`; headline generation; ROUGE-style quality scoring |
| `SemanticSearchEngine` | embedding search — **sentence-transformers** if installed, else an **LSA (TF-IDF + SVD)** embedding that runs anywhere; NL search, nearest-neighbour retrieval, semantic clustering |
| `ContentEnhancer` | collection-level **insight generation** (entities, sentiment climate, topics) and **coverage-gap** detection |

In [5]:
# ============================================================
# NewsBot 2.0 — Part 3: Language Understanding & Generation
# ============================================================
import re
import numpy as np
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize



def _sent_split(text):
    if HAS_NLTK:
        try:
            from nltk.tokenize import sent_tokenize
            return [s.strip() for s in sent_tokenize(str(text)) if s.strip()]
        except Exception:
            pass
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", str(text)) if s.strip()]


# ---------------------------------------------------------------
# 5. Intelligent summarization (extractive TextRank + optional abstractive)
# ---------------------------------------------------------------
class IntelligentSummarizer:
    """Extractive TextRank summariser (always available) with an optional
    abstractive transformer path, headline generation, and overlap-based
    quality scoring."""
    def __init__(self, config=None):
        self.config = config or NewsBot2Config()
        self._abstractive = None
        if HAS_TRANSFORMERS:
            try:
                from transformers import pipeline
                self._abstractive = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")
            except Exception:
                self._abstractive = None

    def _textrank(self, text, n):
        sents = _sent_split(text)
        if len(sents) <= n:
            return sents
        vec = TfidfVectorizer(stop_words="english")
        try:
            tfidf = vec.fit_transform(sents)
        except ValueError:
            return sents[:n]
        sim = cosine_similarity(tfidf)
        np.fill_diagonal(sim, 0.0)
        g = nx.from_numpy_array(sim)
        scores = nx.pagerank(g, max_iter=200)
        ranked = sorted(range(len(sents)), key=lambda i: scores[i], reverse=True)[:n]
        return [sents[i] for i in sorted(ranked)]     # keep original order

    def summarize_article(self, article_text, summary_type="balanced", abstractive=False):
        n = {"brief": 2, "balanced": self.config.summary_sentences, "detailed": 5}.get(summary_type, 3)
        if abstractive and self._abstractive is not None:
            words = str(article_text).split()
            chunk = " ".join(words[:700])
            try:
                out = self._abstractive(chunk, max_length=130, min_length=30, truncation=True)
                return out[0]["summary_text"]
            except Exception:
                pass
        return " ".join(self._textrank(article_text, n))

    def summarize_multiple_articles(self, articles, focus_topic=None, n=4):
        joined = " ".join(articles)
        return self.summarize_article(joined, summary_type="detailed")

    def generate_headlines(self, article_text):
        sents = _sent_split(article_text)
        if not sents:
            return ""
        # most central sentence, trimmed to a headline length
        lead = self._textrank(article_text, 1)[0] if len(sents) > 1 else sents[0]
        words = lead.split()
        return " ".join(words[:12]).rstrip(".,;:") + ("…" if len(words) > 12 else "")

    def assess_summary_quality(self, original_text, summary):
        """Lightweight ROUGE-1 style recall + compression ratio."""
        o = set(preprocess_text(original_text).split())
        s = set(preprocess_text(summary).split())
        recall = len(o & s) / (len(o) + 1e-9)
        compression = len(summary.split()) / (len(original_text.split()) + 1e-9)
        return {"coverage_r1": round(recall, 3), "compression": round(compression, 3)}


# ---------------------------------------------------------------
# 6. Semantic search + similarity + clustering
# ---------------------------------------------------------------
class SemanticSearchEngine:
    """Embedding-based search. Uses sentence-transformers if installed, else an
    LSA (TF-IDF + TruncatedSVD) embedding that runs anywhere. Supports NL search,
    similar-article retrieval, and semantic clustering."""
    def __init__(self, config=None):
        self.config = config or NewsBot2Config()
        self.method = None
        self.sbert = None
        self.vectorizer = None
        self.svd = None
        self.doc_embeddings = None
        self.documents = None
        if HAS_SBERT and self.config.embedding_method in ("auto", "sbert"):
            try:
                from sentence_transformers import SentenceTransformer
                self.sbert = SentenceTransformer("all-MiniLM-L6-v2")
                self.method = "sbert"
            except Exception:
                self.sbert = None
        if self.method is None:
            self.method = "lsa"

    def _embed(self, texts, fit=False):
        if self.method == "sbert":
            return normalize(np.asarray(self.sbert.encode(list(texts), show_progress_bar=False)))
        # LSA path
        clean = [preprocess_text(t) for t in texts]
        if fit:
            self.vectorizer = TfidfVectorizer(max_features=4000, ngram_range=(1, 2))
            tfidf = self.vectorizer.fit_transform(clean)
            k = min(128, tfidf.shape[1] - 1, max(2, tfidf.shape[0] - 1))
            self.svd = TruncatedSVD(n_components=k, random_state=42)
            emb = self.svd.fit_transform(tfidf)
        else:
            tfidf = self.vectorizer.transform(clean)
            emb = self.svd.transform(tfidf)
        return normalize(emb)

    def encode_documents(self, documents):
        self.documents = list(documents)
        self.doc_embeddings = self._embed(self.documents, fit=True)
        return self.doc_embeddings

    def semantic_search(self, query_text, top_k=5):
        assert self.doc_embeddings is not None, "call encode_documents() first"
        q = self._embed([query_text], fit=False)
        sims = cosine_similarity(q, self.doc_embeddings)[0]
        idx = np.argsort(sims)[::-1][:top_k]
        return [{"index": int(i), "score": round(float(sims[i]), 3),
                 "preview": self.documents[i][:160]} for i in idx]

    def find_similar_articles(self, query_index, top_k=5):
        q = self.doc_embeddings[query_index:query_index + 1]
        sims = cosine_similarity(q, self.doc_embeddings)[0]
        idx = [i for i in np.argsort(sims)[::-1] if i != query_index][:top_k]
        return [{"index": int(i), "score": round(float(sims[i]), 3),
                 "preview": self.documents[i][:160]} for i in idx]

    def cluster_similar_content(self, n_clusters=5):
        km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        labels = km.fit_predict(self.doc_embeddings)
        return labels


# ---------------------------------------------------------------
# 7. Content enhancement + insight generation
# ---------------------------------------------------------------
class ContentEnhancer:
    """Generates collection-level insights: dominant entities, category mix,
    sentiment climate, and coverage-gap hints. Wires optional analyzers."""
    def __init__(self, config=None, entity_mapper=None, sentiment_tracker=None,
                 topic_engine=None):
        self.config = config or NewsBot2Config()
        self.entity_mapper = entity_mapper
        self.sentiment_tracker = sentiment_tracker
        self.topic_engine = topic_engine

    def enhance_article(self, article_text):
        out = {"length_words": len(str(article_text).split())}
        if self.entity_mapper is not None:
            ents = self.entity_mapper.extract_entities(article_text)
            out["entities"] = ents[:15]
            out["entity_types"] = dict(__import__("collections").Counter(l for _, l in ents))
        if self.sentiment_tracker is not None:
            out["sentiment"] = self.sentiment_tracker.analyze_sentiment(article_text)
        if self.topic_engine is not None and self.topic_engine.model is not None:
            out["topic"] = self.topic_engine.get_article_topics(article_text)
        return out

    def generate_insights(self, articles):
        from collections import Counter
        insights = {"n_articles": len(articles)}
        if self.sentiment_tracker is not None:
            comps = [self.sentiment_tracker.analyze_sentiment(a)["compound"] for a in articles]
            insights["avg_sentiment"] = round(float(np.mean(comps)), 3)
            insights["sentiment_climate"] = ("positive" if np.mean(comps) > 0.05 else
                                             "negative" if np.mean(comps) < -0.05 else "mixed")
        if self.entity_mapper is not None:
            allents = Counter()
            for a in articles:
                for t, l in self.entity_mapper.extract_entities(a):
                    allents[t] += 1
            insights["top_entities"] = allents.most_common(10)
        return insights

    def detect_information_gaps(self, articles, topic_terms):
        """Which requested topic terms are under-represented in the collection."""
        joined = " ".join(preprocess_text(a) for a in articles)
        counts = {t: joined.count(t.lower()) for t in topic_terms}
        gaps = [t for t, c in counts.items() if c == 0]
        return {"coverage": counts, "gaps": gaps}


In [6]:
# --- Section 3 demo ---
article = df["Text"].iloc[0]

summ = IntelligentSummarizer(config)
s = summ.summarize_article(article, "balanced")
print("Summary:", s[:200])
print("Headline:", summ.generate_headlines(article))
print("Quality:", summ.assess_summary_quality(article, s))

search = SemanticSearchEngine(config)
search.encode_documents(df["Text"].tolist())
print("\nSearch method:", search.method)
for hit in search.semantic_search("football championship winning goal", top_k=3):
    print(f"  ({hit['score']}) {hit['preview'][:80]}")

enhancer = ContentEnhancer(config, ents, senti, topics)
ins = enhancer.generate_insights(df["Text"].tolist())
print("\nInsights:", {k: ins[k] for k in ins if k != "top_entities"})
print("Coverage gaps test:", enhancer.detect_information_gaps(df["Text"].tolist(), ["election", "quantum", "goal"]))

Summary: The striker scored a stunning goal in the final minute to secure victory for his team in the championship match. The tennis champion defeated her opponent in straight sets to claim the grand slam titl
Headline: The striker scored a stunning goal in the final minute to secure…
Quality: {'coverage_r1': 1.0, 'compression': 1.0}

Search method: lsa
  (0.754) The striker scored a stunning goal in the final minute to secure victory for his
  (0.754) The striker scored a stunning goal in the final minute to secure victory for his
  (0.71) The athletics team broke a world record at the championships, thrilling fans who



Insights: {'n_articles': 200, 'avg_sentiment': 0.376, 'sentiment_climate': 'positive'}
Coverage gaps test: {'coverage': {'election': 17, 'quantum': 0, 'goal': 19}, 'gaps': ['quantum']}


## 🌍 Section 4 — Multilingual Intelligence

`MultilingualProcessor` provides **language detection** (`langdetect`, with confidence),
**translation** (`deep_translator`, online — degrades cleanly to an informative message when
offline/uninstalled), **cross-lingual coverage comparison**, and a `process_to_english` helper
that normalises any article to English so the rest of the pipeline is language-agnostic.

In [7]:
# ============================================================
# NewsBot 2.0 — Part 4: Multilingual + Conversational Interface
# ============================================================
import re
import numpy as np
from collections import Counter


# ---------------------------------------------------------------
# 8. Multilingual intelligence
# ---------------------------------------------------------------
class MultilingualProcessor:
    """Language detection (langdetect) + translation (deep_translator, online).
    Degrades gracefully offline: detection still works, translation reports its
    availability instead of crashing. Cross-lingual comparison aggregates
    coverage across language buckets."""
    LANG_NAMES = {"en": "English", "es": "Spanish", "fr": "French", "de": "German",
                  "it": "Italian", "pt": "Portuguese", "nl": "Dutch", "ru": "Russian",
                  "zh-cn": "Chinese", "ar": "Arabic", "hi": "Hindi", "ja": "Japanese"}

    def __init__(self, config=None):
        self.config = config or NewsBot2Config()
        self._translator_ok = HAS_TRANSLATOR
        if HAS_LANGDET:
            from langdetect import DetectorFactory
            DetectorFactory.seed = 0

    def detect_language(self, text):
        text = str(text).strip()
        if not text:
            return {"language": "unknown", "name": "Unknown", "confidence": 0.0}
        if HAS_LANGDET:
            try:
                from langdetect import detect_langs
                best = detect_langs(text)[0]
                return {"language": best.lang,
                        "name": self.LANG_NAMES.get(best.lang, best.lang),
                        "confidence": round(float(best.prob), 3)}
            except Exception:
                pass
        # crude fallback: stopword overlap
        return {"language": "en", "name": "English (assumed)", "confidence": 0.3}

    def translate_text(self, text, target_language="en", source_language="auto"):
        if not self._translator_ok:
            return {"translated": None, "ok": False,
                    "note": "deep_translator not installed; run pip install deep-translator (needs internet)."}
        try:
            from deep_translator import GoogleTranslator
            out = GoogleTranslator(source=source_language, target=target_language).translate(str(text)[:4900])
            return {"translated": out, "ok": True, "target": target_language}
        except Exception as e:
            return {"translated": None, "ok": False, "note": f"translation failed: {e}"}

    def analyze_cross_lingual(self, articles_by_language):
        """articles_by_language: {lang_code: [texts]} -> coverage comparison."""
        summary = {}
        for lang, arts in articles_by_language.items():
            words = sum(len(str(a).split()) for a in arts)
            summary[lang] = {"name": self.LANG_NAMES.get(lang, lang),
                             "articles": len(arts),
                             "avg_words": round(words / max(1, len(arts)), 1)}
        return summary

    def process_to_english(self, text):
        """Detect -> translate to English if needed. Returns unified English text."""
        det = self.detect_language(text)
        if det["language"] == "en":
            return {"text_en": str(text), "source_language": "en", "translated": False}
        tr = self.translate_text(text, "en")
        return {"text_en": tr.get("translated") or str(text),
                "source_language": det["language"],
                "translated": bool(tr.get("ok"))}


# ---------------------------------------------------------------
# 9. Conversational interface (intent + slot filling + response)
# ---------------------------------------------------------------
class ConversationalInterface:
    """Natural-language front end. Rule+keyword intent classifier, entity/param
    slot extraction, execution against the integrated system, response
    generation, and context carry-over for follow-ups."""
    INTENTS = {
        "search":    ["find", "show", "search", "articles about", "news about", "look for"],
        "summarize": ["summarize", "summary", "tl;dr", "brief me", "sum up"],
        "sentiment": ["sentiment", "mood", "positive", "negative", "feeling", "opinion"],
        "classify":  ["classify", "category", "what kind", "which topic", "categorize"],
        "compare":   ["compare", "versus", " vs ", "difference between"],
        "entities":  ["who is", "connection", "related to", "entities", "people in", "link between"],
        "trend":     ["trend", "over time", "evolution", "changing", "trending"],
        "topics":    ["topics", "themes", "what topics", "discover"],
    }

    def __init__(self, newsbot_system=None, config=None):
        self.newsbot = newsbot_system
        self.config = config or NewsBot2Config()
        self.history = []

    def classify_intent(self, user_query):
        q = " " + user_query.lower() + " "
        first_word = user_query.strip().lower().split()[0] if user_query.strip() else ""
        scores = {}
        for intent, kws in self.INTENTS.items():
            s = sum(kw in q for kw in kws)
            # leading-verb bonus: an imperative like "Summarize ..." should win
            # ties against incidental keywords ("articles about") elsewhere.
            if any(kw.strip() == first_word for kw in kws):
                s += 2
            scores[intent] = s
        best = max(scores, key=scores.get)
        if scores[best] == 0:
            return {"intent": "search", "confidence": 0.3}   # sensible default
        total = sum(scores.values())
        return {"intent": best, "confidence": round(scores[best] / total, 2)}

    def extract_query_entities(self, user_query):
        q = user_query.lower()
        params = {}
        cats = ["business", "tech", "technology", "sport", "sports", "politics", "entertainment"]
        found = [c for c in cats if c in q]
        if found:
            params["category"] = found[0].replace("technology", "tech").replace("sports", "sport")
        for pol in ["positive", "negative", "neutral"]:
            if pol in q: params["sentiment"] = pol
        for tf in ["today", "week", "month", "year"]:
            if tf in q: params["timeframe"] = tf
        # quoted phrase or capitalised names as free query
        m = re.findall(r'"([^"]+)"', user_query)
        if m: params["query_text"] = m[0]
        return params

    def process_query(self, user_query, conversation_context=None):
        intent = self.classify_intent(user_query)
        params = self.extract_query_entities(user_query)
        # carry context for follow-ups ("what about last month?")
        if conversation_context:
            for k, v in conversation_context.items():
                params.setdefault(k, v)
        result = self._execute(intent["intent"], params, user_query)
        response = self.generate_response(result, intent["intent"], params)
        self.history.append({"query": user_query, "intent": intent["intent"], "params": params})
        return {"intent": intent, "params": params, "result": result, "response": response}

    def _execute(self, intent, params, raw_query):
        nb = self.newsbot
        if nb is None:
            return {"note": "no system attached (demo mode)"}
        try:
            if intent == "search":
                q = params.get("query_text", raw_query)
                return {"hits": nb.search_engine.semantic_search(q, top_k=3)}
            if intent == "summarize":
                q = params.get("query_text", raw_query)
                hits = nb.search_engine.semantic_search(q, top_k=3)
                joined = " ".join(nb.search_engine.documents[h["index"]] for h in hits)
                return {"summary": nb.summarizer.summarize_article(joined, "balanced")}
            if intent == "classify":
                return {"classification": nb.classifier.predict_with_confidence(raw_query)}
            if intent == "sentiment":
                q = params.get("query_text", raw_query)
                hits = nb.search_engine.semantic_search(q, top_k=5)
                comps = [nb.sentiment_tracker.analyze_sentiment(
                    nb.search_engine.documents[h["index"]])["compound"] for h in hits]
                return {"avg_sentiment": round(float(np.mean(comps)), 3) if comps else 0.0}
            if intent == "topics":
                return {"topics": {t: nb.topic_engine.top_words(t, 5)
                                   for t in range(nb.topic_engine.n_topics)}}
            if intent == "entities":
                return {"entities": nb.entity_mapper.extract_entities(raw_query)[:10]}
            if intent == "trend":
                return {"note": "trend tracking requires dated corpus (see track_topic_trends)"}
            if intent == "compare":
                return {"note": "comparison runs search on each subject then contrasts sentiment/topics"}
        except Exception as e:
            return {"error": str(e)}
        return {"note": "handled as generic search"}

    def generate_response(self, result, intent, params):
        if "error" in result:
            return f"I hit a problem running that ({result['error']}). Could you rephrase?"
        if intent == "search" and "hits" in result:
            if not result["hits"]:
                return "I couldn't find matching articles."
            lines = [f"Top {len(result['hits'])} matches:"]
            for h in result["hits"]:
                lines.append(f"  • ({h['score']}) {h['preview'][:90]}…")
            return "\n".join(lines)
        if intent == "summarize" and "summary" in result:
            return "Here's a summary of the most relevant coverage:\n" + result["summary"]
        if intent == "classify" and "classification" in result:
            c = result["classification"]
            return f"That reads as **{c['primary']}** ({c['confidence']:.0%} confidence)."
        if intent == "sentiment":
            v = result.get("avg_sentiment", 0)
            mood = "positive" if v > 0.05 else "negative" if v < -0.05 else "neutral"
            return f"Overall sentiment for that query is {mood} (avg compound {v})."
        if intent == "topics" and "topics" in result:
            lines = ["Discovered topics:"]
            for t, words in result["topics"].items():
                lines.append(f"  Topic {t}: {', '.join(words)}")
            return "\n".join(lines)
        if intent == "entities" and "entities" in result:
            if not result["entities"]:
                return "I didn't detect named entities in that."
            return "Entities: " + ", ".join(f"{t} ({l})" for t, l in result["entities"])
        return str(result.get("note", "Done."))

    def handle_follow_up(self, follow_up_query, ):
        ctx = self.history[-1]["params"] if self.history else {}
        return self.process_query(follow_up_query, conversation_context=ctx)


In [8]:
# --- Section 4 demo ---
mult = MultilingualProcessor(config)
for t in ["The government announced a new economic policy today.",
          "El gobierno anunció una nueva política económica hoy.",
          "Le gouvernement a annoncé une nouvelle politique économique."]:
    print(mult.detect_language(t), "->", t[:45])
print("Translate demo:", mult.translate_text("Hola, ¿cómo estás?", "en"))
print("Cross-lingual:", mult.analyze_cross_lingual({"en": df["Text"].tolist()[:20]}))

{'language': 'en', 'name': 'English', 'confidence': 1.0} -> The government announced a new economic polic
{'language': 'es', 'name': 'Spanish', 'confidence': 1.0} -> El gobierno anunció una nueva política económ
{'language': 'fr', 'name': 'French', 'confidence': 1.0} -> Le gouvernement a annoncé une nouvelle politi
Translate demo: {'translated': None, 'ok': False, 'note': 'deep_translator not installed; run pip install deep-translator (needs internet).'}
Cross-lingual: {'en': {'name': 'English', 'articles': 20, 'avg_words': 35.6}}


## 💬 Section 5 — Conversational Interface

`ConversationalInterface` turns natural-language queries into actions. It performs
**intent classification** (search / summarize / sentiment / classify / compare / entities /
trend / topics) with a leading-verb tie-breaker, **slot extraction** (category, sentiment,
timeframe, quoted query), **executes** against the live system, and **generates responses**.
`handle_follow_up` carries context so *"what about last month?"* resolves against the prior turn.

*(The class is defined in Section 4's cell alongside the multilingual processor; it is
exercised through the integrated system in Section 6.)*

## 🔧 Section 6 — System Integration & Testing

`NewsBot2IntegratedSystem` wires every component together. `NewsBot2TestSuite` runs
**component**, **integration**, and **edge-case** tests (empty / whitespace / very-short /
non-English inputs) so robustness is demonstrable, not assumed.

In [9]:
# ============================================================
# NewsBot 2.0 — Part 5: Integration, Testing, Evaluation
# ============================================================
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split



# ---------------------------------------------------------------
# 10. Integrated system orchestrator
# ---------------------------------------------------------------
class NewsBot2IntegratedSystem:
    """Wires every component into one system. `fit` trains/index everything on a
    corpus; `comprehensive_analysis` runs the full pipeline on one article;
    `query_interface` routes NL queries through the conversational layer."""
    def __init__(self, config=None):
        self.config = config or NewsBot2Config()
        self.classifier = AdvancedNewsClassifier(self.config)
        self.topic_engine = TopicDiscoveryEngine(self.config.n_topics,
                                                 self.config.topic_method, self.config)
        self.sentiment_tracker = SentimentEvolutionTracker(self.config)
        self.entity_mapper = EntityRelationshipMapper(self.config)
        self.summarizer = IntelligentSummarizer(self.config)
        self.search_engine = SemanticSearchEngine(self.config)
        self.multilingual = MultilingualProcessor(self.config)
        self.enhancer = ContentEnhancer(self.config, self.entity_mapper,
                                        self.sentiment_tracker, self.topic_engine)
        self.conversation = ConversationalInterface(self, self.config)
        self._fitted = False

    def fit(self, texts, labels=None, build_graph_on=200):
        texts = list(texts)
        self.corpus_texts = texts
        self.corpus_labels = list(labels) if labels is not None else None
        if labels is not None:
            self.classifier.train(texts, list(labels))
        self.topic_engine.fit_topics(texts)
        self.search_engine.encode_documents(texts)
        self.entity_mapper.build_knowledge_graph(texts[:build_graph_on])
        self._fitted = True
        return self

    def comprehensive_analysis(self, article_text):
        # multilingual normalisation first
        norm = self.multilingual.process_to_english(article_text)
        text = norm["text_en"]
        results = {
            "language": {"source": norm["source_language"], "translated": norm["translated"]},
            "classification": self.classifier.predict_with_confidence(text)
                              if self.classifier._fitted else None,
            "sentiment": self.sentiment_tracker.analyze_sentiment(text),
            "entities": self.entity_mapper.extract_entities(text)[:15],
            "topics": self.topic_engine.get_article_topics(text)
                      if self.topic_engine.model is not None else None,
            "summary": self.summarizer.summarize_article(text, "balanced"),
            "headline": self.summarizer.generate_headlines(text),
        }
        return results

    def batch_analysis(self, articles, limit=None):
        out = []
        for i, a in enumerate(articles[:limit] if limit else articles):
            try:
                out.append(self.comprehensive_analysis(a))
            except Exception as e:
                out.append({"error": str(e), "index": i})
        return out

    def query_interface(self, user_query, context=None):
        return self.conversation.process_query(user_query, context)

    def generate_insights_report(self, articles, report_type="comprehensive"):
        base = self.enhancer.generate_insights(articles)
        base["report_type"] = report_type
        if self.topic_engine.model is not None:
            base["topics"] = {t: self.topic_engine.top_words(t, 6)
                              for t in range(self.topic_engine.n_topics)}
        return base


# ---------------------------------------------------------------
# 11. Testing framework
# ---------------------------------------------------------------
class NewsBot2TestSuite:
    """Component, integration, and edge-case tests returning pass/fail dicts."""
    def __init__(self, system):
        self.nb = system

    def test_individual_components(self, sample_text):
        r = {}
        r["classifier"] = self.nb.classifier._fitted and \
            "primary" in self.nb.classifier.predict_with_confidence(sample_text)
        r["topics"] = self.nb.topic_engine.model is not None and \
            "dominant_topic" in self.nb.topic_engine.get_article_topics(sample_text)
        r["sentiment"] = "compound" in self.nb.sentiment_tracker.analyze_sentiment(sample_text)
        r["ner"] = isinstance(self.nb.entity_mapper.extract_entities(sample_text), list)
        r["summarizer"] = len(self.nb.summarizer.summarize_article(sample_text)) > 0
        r["search"] = len(self.nb.search_engine.semantic_search(sample_text, 1)) > 0
        r["multilingual"] = "language" in self.nb.multilingual.detect_language(sample_text)
        return r

    def test_integration(self, sample_text):
        r = {}
        try:
            a = self.nb.comprehensive_analysis(sample_text)
            r["comprehensive_analysis"] = all(k in a for k in
                ["classification", "sentiment", "entities", "summary"])
        except Exception as e:
            r["comprehensive_analysis"] = False; r["error"] = str(e)
        try:
            q = self.nb.query_interface("Show me sport news")
            r["query_interface"] = "response" in q
        except Exception as e:
            r["query_interface"] = False; r["error2"] = str(e)
        return r

    def test_edge_cases(self):
        r = {}
        for name, val in [("empty", ""), ("whitespace", "   "),
                          ("very_short", "Hi."), ("non_english", "Bonjour le monde.")]:
            try:
                self.nb.comprehensive_analysis(val)
                r[name] = True
            except Exception as e:
                r[name] = f"FAILED: {e}"
        return r

    def run_all(self, sample_text):
        return {"components": self.test_individual_components(sample_text),
                "integration": self.test_integration(sample_text),
                "edge_cases": self.test_edge_cases()}


# ---------------------------------------------------------------
# 12. Evaluation framework
# ---------------------------------------------------------------
class NewsBot2Evaluator:
    """Quantitative evaluation across components with a consolidated report."""
    def __init__(self, system):
        self.nb = system

    def evaluate_classification(self, texts, labels):
        return self.nb.classifier.evaluate(list(texts), list(labels))

    def evaluate_topic_quality(self, documents):
        return {"coherence_c_v": self.nb.topic_engine.coherence(documents),
                "n_topics": self.nb.topic_engine.n_topics}

    def evaluate_summarization(self, articles, n=10):
        scores = []
        for a in articles[:n]:
            s = self.nb.summarizer.summarize_article(a, "balanced")
            scores.append(self.nb.summarizer.assess_summary_quality(a, s))
        return {"avg_coverage_r1": round(np.mean([s["coverage_r1"] for s in scores]), 3),
                "avg_compression": round(np.mean([s["compression"] for s in scores]), 3)}

    def evaluate_search(self, n_probes=20):
        """Retrieval sanity check on the INDEXED corpus: does the nearest
        neighbour share the probe article's category? Uses labels aligned to
        the search engine's own documents (stored on the system at fit time)."""
        labels = getattr(self.nb, "corpus_labels", None)
        docs = self.nb.search_engine.documents
        if labels is None or docs is None:
            return {"top1_category_match": None, "note": "no labelled indexed corpus"}
        n = len(docs)
        hits = 0
        idxs = np.random.RandomState(42).choice(n, min(n_probes, n), replace=False)
        for i in idxs:
            res = self.nb.search_engine.find_similar_articles(int(i), top_k=1)
            if res and labels[res[0]["index"]] == labels[i]:
                hits += 1
        return {"top1_category_match": round(hits / len(idxs), 3)}

    def generate_evaluation_report(self, texts, labels):
        report = {}
        report["classification"] = self.evaluate_classification(texts, labels)
        report["topics"] = self.evaluate_topic_quality(list(texts))
        report["summarization"] = self.evaluate_summarization(list(texts))
        report["search"] = self.evaluate_search()
        return report


In [10]:
# --- Build the full system ---
newsbot = NewsBot2IntegratedSystem(config).fit(Xtr.tolist(), ytr.tolist())

sample = "The striker scored a dramatic late goal to win the championship final for his club."
analysis = newsbot.comprehensive_analysis(sample)
print("Classification:", analysis["classification"]["primary"],
      f"({analysis['classification']['confidence']:.0%})")
print("Sentiment:", analysis["sentiment"]["label"])
print("Topic:", analysis["topics"]["label"])
print("Headline:", analysis["headline"])

print("\n--- Conversational queries ---")
for q in ["Show me business news",
          "Summarize articles about technology",
          "classify this: the minister announced a new tax policy",
          "What is the sentiment of sport coverage?"]:
    print("Q:", q)
    print("  ", newsbot.query_interface(q)["response"].split("\n")[0])

Classification: sport (74%)
Sentiment: positive
Topic: set, film, surprise
Headline: The striker scored a dramatic late goal to win the championship final…

--- Conversational queries ---
Q: Show me business news
   Top 3 matches:
Q: Summarize articles about technology
   Here's a summary of the most relevant coverage:
Q: classify this: the minister announced a new tax policy
   That reads as **politics** (61% confidence).
Q: What is the sentiment of sport coverage?
   Overall sentiment for that query is positive (avg compound 0.527).


In [11]:
# --- Testing framework ---
suite = NewsBot2TestSuite(newsbot)
results = suite.run_all(sample)
import json
print(json.dumps(results, indent=2, default=str))

{
  "components": {
    "classifier": true,
    "topics": true,
    "sentiment": true,
    "ner": true,
    "summarizer": true,
    "search": true,
    "multilingual": true
  },
  "integration": {
    "comprehensive_analysis": true,
    "query_interface": true
  },
  "edge_cases": {
    "empty": true,
    "whitespace": true,
    "very_short": true,
    "non_english": true
  }
}


## 📈 Section 7 — Evaluation & Documentation

`NewsBot2Evaluator` produces quantitative metrics per component: classification
accuracy / macro-F1, topic **coherence (c_v)**, summarization coverage & compression, and a
retrieval sanity check (does the nearest neighbour share the probe's category?).

In [12]:
# --- Full evaluation report ---
evaluator = NewsBot2Evaluator(newsbot)
report = evaluator.generate_evaluation_report(Xte.tolist(), yte.tolist())

print("CLASSIFICATION  acc =", report["classification"]["accuracy"],
      "| macro-F1 =", report["classification"]["macro_f1"])
print("TOPICS          coherence_c_v =", report["topics"]["coherence_c_v"])
print("SUMMARIZATION  ", report["summarization"])
print("SEARCH         ", report["search"])
print("\nPer-class report:\n", report["classification"]["report"])

CLASSIFICATION  acc = 1.0 | macro-F1 = 1.0
TOPICS          coherence_c_v = 0.8082
SUMMARIZATION   {'avg_coverage_r1': np.float64(1.0), 'avg_compression': np.float64(1.0)}
SEARCH          {'top1_category_match': 1.0}

Per-class report:
                precision    recall  f1-score   support

     business       1.00      1.00      1.00         8
entertainment       1.00      1.00      1.00         8
     politics       1.00      1.00      1.00         8
        sport       1.00      1.00      1.00         8
         tech       1.00      1.00      1.00         8

     accuracy                           1.00        40
    macro avg       1.00      1.00      1.00        40
 weighted avg       1.00      1.00      1.00        40



## ✅ Requirements Checklist — where each is implemented

| Requirement | Class · method |
|---|---|
| Multi-class classification + confidence | `AdvancedNewsClassifier.predict_with_confidence` |
| Prediction explanation | `AdvancedNewsClassifier.explain_prediction` |
| Topic modeling (LDA/NMF) | `TopicDiscoveryEngine.fit_topics` |
| Topic coherence + trends | `TopicDiscoveryEngine.coherence` · `track_topic_trends` |
| Sentiment + temporal tracking | `SentimentEvolutionTracker.track_sentiment_over_time` |
| Sentiment anomaly detection | `SentimentEvolutionTracker.detect_sentiment_anomalies` |
| Entity relationship / knowledge graph | `EntityRelationshipMapper.build_knowledge_graph` · `find_entity_connections` |
| Summarization (extractive + abstractive) | `IntelligentSummarizer.summarize_article` |
| Headline generation + quality | `IntelligentSummarizer.generate_headlines` · `assess_summary_quality` |
| Semantic search (embeddings) | `SemanticSearchEngine.semantic_search` |
| Semantic clustering | `SemanticSearchEngine.cluster_similar_content` |
| Content enhancement + insight gen | `ContentEnhancer.generate_insights` · `detect_information_gaps` |
| Language detection + confidence | `MultilingualProcessor.detect_language` |
| Translation + quality note | `MultilingualProcessor.translate_text` |
| Cross-lingual analysis | `MultilingualProcessor.analyze_cross_lingual` |
| Intent classification | `ConversationalInterface.classify_intent` |
| NL query processing + slots | `ConversationalInterface.process_query` · `extract_query_entities` |
| Context-aware follow-ups | `ConversationalInterface.handle_follow_up` |
| Integration | `NewsBot2IntegratedSystem` |
| Testing framework | `NewsBot2TestSuite` |
| Evaluation framework | `NewsBot2Evaluator` |

### 🔌 Enabling the optional upgrades
```bash
pip install transformers sentence-transformers   # abstractive summaries, SBERT search
pip install deep-translator                       # live translation (needs internet)
pip install gensim                                # topic coherence (c_v)
python -m spacy download en_core_web_sm           # NER (if not already present)
```
When present, these are detected automatically at the top of the notebook and the relevant
components upgrade with **no code changes**.

### 📦 Deliverables & naming reminder
- GitHub repo (enhanced from midterm)
- `FP_TechnicalDoc_[Name]_[GroupName]_ITAI2373.pdf`
- `FP_ExecutiveSummary_[Name]_[GroupName]_ITAI2373.pdf`
- `FP_Presentation_[Name]_[GroupName]_ITAI2373.pptx` (or `FP_VideoPresentation_…mp4`)
- `FP_ReflectiveJournal_[GroupName]_ITAI2373.pdf`